# **Mount Drive**

# **Load Data**

In [39]:
import xarray as xr
import pandas as pd
import numpy as np
import geopandas as gpd
from matplotlib import pyplot as plt
import seaborn as sns
from glob import glob

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)
warnings.simplefilter(action='ignore', category=RuntimeWarning)



In [ ]:
import pandas as pd
import numpy as np
fpath_water = '<DATA_ROOT>/WaterUsage/WaterUse_crop_2009_2023.csv'
dfw = pd.read_csv(fpath_water)
# dfw = dfw.drop(columns=['STATE' ,	'COUNTY']).rename(columns={'Year':'year'})
# dfw['IC-WGWFr'] = pd.to_numeric(dfw['IC-WGWFr'], errors='coerce')
# dfw['IC-WSWFr'] = pd.to_numeric(dfw['IC-WSWFr'], errors='coerce')
# dfw['IC-WFrTo'] = pd.to_numeric(dfw['IC-WFrTo'], errors='coerce')
# dfw = dfw.dropna()
# formatted_numbers = [str(n).zfill(5) for n in dfw['FIPS'].values]
# dfw['FIPS'] = formatted_numbers
# # dfw = dfw[dfw['IC-WFrTo'] > 0]


In [ ]:
ffail = '<DATA_ROOT>/crop_failure_data/crops_failure_0595.csv'
df_fail = pd.read_csv(ffail)
df_fail = df_fail.drop(columns=['Unnamed: 0'])
df_fail = df_fail.sort_values(by=['FIPS'])
# cond1 = df_fail['year']>2011
cond2 = df_fail['Planted Acres'] > 0
df_fail = df_fail[cond2]
df_fail['fail_share'] = df_fail['Failed Acres'] /  df_fail['Planted Acres']
df_fail = df_fail[df_fail.fail_share < 1]
df_fail = df_fail[df_fail.fail_share > 0]

formatted_numbers = [str(n).zfill(5) for n in df_fail['FIPS'].values]
df_fail['FIPS'] = formatted_numbers
df_fail
# df_fail.year.unique()

In [ ]:
fsvi = '<DATA_ROOT>/SVI/SVI_timeseries.csv'
df_svi = pd.read_csv(fsvi)
df_svi = df_svi.sort_values(by=['FIPS'])
formatted_numbers = [str(n).zfill(5) for n in df_svi['FIPS'].values]
df_svi['FIPS'] = formatted_numbers
df_svi.iloc[0:5,:]

In [ ]:
fpath1 = '<DATA_ROOT>/WeatherIndex/Temp_Tercile_11nov.csv'
dft = pd.read_csv(fpath1)
dft = dft.rename(columns={'Year':'year'})
print(dft.iloc[0:2,:])

fpath2 = '<DATA_ROOT>/WeatherIndex/Drought_Map.csv'
dfd = pd.read_csv(fpath2)
dfd = dfd.rename(columns={'Year':'year', 'STATE':'State', 'COUNTY':'County'})
fpath = '<DATA_ROOT>/WeatherIndex/svi2020fips.csv'
df_fips_obj = pd.read_csv(fpath)[['FIPS',	'OBJECTID']]
dfd = pd.merge(dfd,df_fips_obj,on=['OBJECTID'])
print(dfd.iloc[0:2,:])

In [ ]:
drought_indices = ['PDSI', 'SPEI3', 'SPEI6', 'SPEI9', 'SPEI12', 'SPI3', 'SPI6', 'SPI9', 'SPI12']
for idx in drought_indices:
  dfd.loc[dfd[idx] == 0,idx] = 1
print(dfd.iloc[0:2,:])

HW_indices = ['Thr26', 'Thr27', 'Thr28', 'Thr29', 'Thr30', 'Thr31', 'Thr32', 'Thr33', 'Thr34', 'Thr35', 'Thr36']
for idx in HW_indices:
  dft.loc[dft[idx] == 0,idx] = 1
print(dft.iloc[0:2,:])


In [ ]:
shp_counties = '<DATA_ROOT>/SVI/Counties/'
counties = gpd.read_file(shp_counties)
# formatted_numbers = [int(n) for n in counties['FIPS'].values]
# counties['FIPS'] = formatted_numbers
counties.iloc[0:5,:]

In [ ]:
ClimReg_dir = '<DATA_ROOT>/US_ClimateRegions/US_ClimateRegions/'
gdf_clim = gpd.read_file(ClimReg_dir)
gdf_clim

In [ ]:
# import zipfile
# zip_path = '<DATA_ROOT>/CONUS_States/CONUS_STATES.zip'
# extract_dir = '<DATA_ROOT>/CONUS_States/'
# with zipfile.ZipFile(zip_path, 'r') as zip_ref:
#     zip_ref.extractall(extract_dir)

In [ ]:
states_dir = '<DATA_ROOT>/CONUS_States/'
gdf_states = gpd.read_file(states_dir)
gdf_states

# **Annual Groundwater Usage**

In [ ]:
import pandas as pd
import numpy as np
fpath_water = '<DATA_ROOT>/WaterUsage/WaterUse_crop_v3.csv'
dfw = pd.read_csv(fpath_water)
dfw.iloc[:4]
dfw = dfw[['FIPS','IR-WGWFr','IR-WSWFr','IR-WFrTo','IR-IrTot','Year']]
print(dfw.shape)
dfw = dfw.dropna()
dfw

In [ ]:
dfw = dfw.rename(columns={'Year':'year'})
formatted_numbers = [str(n).zfill(5) for n in dfw['FIPS'].values]
dfw['FIPS'] = formatted_numbers

# dfw = dfw[dfw['IC-WFrTo'] > 0]
dfw.loc[:,'gr_wu'] = dfw['IR-WGWFr'] / dfw['IR-WFrTo']
# dfw.loc[:,'gr_wu'] = dfw['IC-WGWFr'] / dfw['IC-WFrTo']
# dfw.loc[:,'wu_rate'] = dfw['IR-WGWFr'] / dfw['IR-IrTot']
# dfw.loc[:,'gr_wu'] = dfw['IR-WGWFr'] / dfw['IR-WFrTo']


In [ ]:
dfw_cnt_mean = dfw.groupby('FIPS').mean().reset_index()[['FIPS','wu_rate']]
cnt_wu_rate = pd.merge(counties,dfw_cnt_mean,on=['FIPS'])
cnt_wu_rate.plot(column='wu_rate',cmap='Reds',vmax=cnt_wu_rate.quantile(0.95).wu_rate)

In [ ]:
dfw_cnt_mean = dfw.groupby('FIPS').mean().reset_index()[['FIPS','gr_wu']]
cnt_wu_rate = pd.merge(counties,dfw_cnt_mean,on=['FIPS'])
cnt_wu_rate.plot(column='gr_wu',cmap='Reds',vmax=cnt_wu_rate.quantile(0.95).gr_wu,)#

In [ ]:
df_svi_cat = df_svi.drop(columns=['State', 'County'])
df_svi_cat = df_svi_cat.set_index(['FIPS','year'])
bins = list(np.arange(0,101)/100)
labels = list(np.arange(0,100)/100)
# bins = [0, 0.25, 0.5, 0.75, 1]
# labels = [1, 2, 3, 4]
df_svi_cat = df_svi_cat.apply(lambda x: pd.cut(x, bins=bins, labels=labels), axis=0)
df_svi_cat = df_svi_cat.reset_index()[['FIPS','year','RPL_THEMES']]
df_svi_cat

In [ ]:
dfw.loc[46610,'gr_wu']

0.0

In [ ]:
dfwC = dfw[['FIPS','year','wu_rate']]
dfwC = dfwC.set_index(['FIPS','year'])
# bins = list(np.arange(0,101,6)/100)
bins = [0, 0.33, 0.66, 1]
quants = dfwC.quantile(bins)
# labels = [0, 0.33, 0.66]
bins = list(quants['wu_rate'])
labels = list(quants['wu_rate'])[0:-1]
dfwC = dfwC.apply(lambda x: pd.cut(x, bins=bins, labels=labels))
dfwC = dfwC.fillna(0)
dfwC = dfwC.reset_index()#[['FIPS','year','wu_rate']]
dfwC

In [ ]:
df_fail = df_fail[['FIPS','Crop','Irrigation Practice','Planted Acres','year','fail_share']]
cond_fail = df_fail['fail_share'] > 0
# cond_fail = True
cond_irig = df_fail['Irrigation Practice'] == 'I'
cols = ['FIPS','Crop','year','Planted Acres','fail_share']
df_merge = pd.merge(df_fail.loc[(cond_fail) & (cond_irig),cols],dfwC,on=['FIPS','year'])
df_merge = pd.merge(df_merge,df_svi_cat,on=['FIPS','year'])
df_merge.iloc[:3]

In [ ]:

df_pivot

In [ ]:
df_group = df_merge.groupby(['RPL_THEMES','wu_rate']).agg({'fail_share':'mean','Planted Acres':'sum'})
df_group.columns = df_group.columns.map(''.join).str.strip()
df_group = df_group.reset_index()
df_pivot = df_group.pivot(index='RPL_THEMES',columns='wu_rate',values='fail_share')
df_pivot = df_pivot.rolling(window=10,min_periods=3).mean()
df_pivot = df_pivot.reset_index()
# df_group = df_group[['RPL_THEMES','year','gr_wu']]
# df_pivot = df_group.pivot(index='year',columns='RPL_THEMES',values='gr_wu')
# df_pivot = df_pivot.rolling(window=4,min_periods=1).mean()
# df_pivot = df_pivot.reset_index()

In [ ]:
import seaborn as sns
fig , ax = plt.subplots(figsize=(10, 6))#,dpi=1200
# sns.lineplot(data=df_group_all, x='year', y='fail_share', color='#969696', ax=ax, linewidth=3, label='All')
sns.lineplot(data=df_pivot,     x='RPL_THEMES', y=df_pivot[0.0],  color='#fecc5c', ax=ax, linewidth=2, label='Low')
# sns.lineplot(data=df_pivot,     x='wu_rate', y=df_pivot[2],  color='#fd8d3c', ax=ax, linewidth=2, label='Low-Medium')
# sns.lineplot(data=df_pivot,     x='wu_rate', y=df_pivot[3],  color='#f03b20', ax=ax, linewidth=2, label='Medium-High')
sns.lineplot(data=df_pivot,     x='RPL_THEMES', y=df_pivot[0.125],  color='#f03b20', ax=ax, linewidth=2, label='Medium')
sns.lineplot(data=df_pivot,     x='RPL_THEMES', y=df_pivot[0.5],  color='#bd0026', ax=ax, linewidth=2, label='High')

# ax.set_xlabel('Year')
# ax.set_ylabel('Groundwater Usage')
# ax.set_xlim(2009,2023)
# plt.title('Annual Groundwater Usage')

plt.legend()

# out_path = '<DATA_ROOT>/Final_Exports/20231126-Failure_Inequality_Plots/'
# plt.savefig(out_path + 'Failure_Share_Intensity_Annual_Timeseries.png',dpi=1200)


# **Timeseries GW**

In [ ]:
import pandas as pd
import numpy as np
fpath_water = '<DATA_ROOT>/WaterUsage/WaterUse_crop_v3.csv'
dfw = pd.read_csv(fpath_water).rename(columns={'Year':'year'})
dfw = dfw[['FIPS','IR-WGWFr','IR-WSWFr','IR-WFrTo','IR-IrTot','year']]
dfw['IR-WGWFr'] = pd.to_numeric(dfw['IR-WGWFr'], errors='coerce')
dfw['IR-IrTot'] = pd.to_numeric(dfw['IR-IrTot'], errors='coerce')

dfw = dfw[ dfw['IR-WGWFr'] > 0]
dfw.loc[:,'gwu_rate'] = dfw['IR-WGWFr'] / dfw['IR-IrTot']
formatted_numbers = [str(n).zfill(5) for n in dfw['FIPS'].values]
dfw['FIPS'] = formatted_numbers
dfw = dfw[['FIPS','year','gwu_rate']]

In [ ]:
df_svi_cat = df_svi.drop(columns=['State','County']).set_index(['FIPS','year'])
bins = [0, 0.25, 0.5, 0.75, 1]
labels = [1, 2, 3, 4]
df_svi_cat = df_svi_cat.apply(lambda x: pd.cut(x, bins=bins, labels=labels), axis=0)
df_svi_cat = df_svi_cat['RPL_THEMES'].reset_index()
df_svi_cat

In [ ]:
df_merge = pd.merge(dfw,df_svi_cat,on=['FIPS','year']).drop(columns=['FIPS'])
df_merge.replace([np.inf, -np.inf], np.nan, inplace=True)
df_merge.dropna(inplace=True)
df_group = df_merge.groupby(['RPL_THEMES','year']).mean().reset_index()
# df_merge[df_merge.year==2009].sort_values('RPL_THEMES')

df_pivot = df_group.pivot(index='year',columns='RPL_THEMES',values='gwu_rate')
df_pivot.plot()

# **Box Plot & Swarm Plot & Histogram**

In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
import geopandas as gpd
from matplotlib import pyplot as plt
import seaborn as sns
from glob import glob

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)
warnings.simplefilter(action='ignore', category=RuntimeWarning)


In [ ]:
import pandas as pd
import numpy as np
fpath_water = '<DATA_ROOT>/WaterUsage/WaterUse_crop_v3.csv'
dfw = pd.read_csv(fpath_water).rename(columns={'Year':'year'})
dfw = dfw[['FIPS','IR-WGWFr','IR-WSWFr','IR-WFrTo','IR-IrTot','year']]
dfw['IR-WGWFr'] = pd.to_numeric(dfw['IR-WGWFr'], errors='coerce')
dfw['IR-WSWFr'] = pd.to_numeric(dfw['IR-WSWFr'], errors='coerce')
dfw['IR-IrTot'] = pd.to_numeric(dfw['IR-IrTot'], errors='coerce')
dfw = dfw[dfw['IR-IrTot'] > 0]
dfw.loc[:,'wu_ratio'] = (dfw['IR-WGWFr'] > dfw['IR-WSWFr']).astype(int)
dfw.loc[:,'gwu_rate'] = dfw['IR-WGWFr'] / dfw['IR-IrTot']
dfw.loc[:,'swu_rate'] = dfw['IR-WSWFr'] / dfw['IR-IrTot']
formatted_numbers = [str(n).zfill(5) for n in dfw['FIPS'].values]
dfw['FIPS'] = formatted_numbers
dfw = dfw[['FIPS','year','gwu_rate','swu_rate','wu_ratio']]
dfw

In [ ]:
def dfw_cat(df , col, G_or_S):
  df = df[df['wu_ratio'] == G_or_S]
  cond = df[col] > 0
  df_cat = df[cond].set_index(['FIPS','year'])[col]
  df_cat = df_cat.to_frame()
  df_cat = df_cat.sort_values(col)
  # bins = [0, 0.2, 0.4, 0.6, 0.8, 1]
  bins = [0, 0.33, 0.66, 1]
  quants = df_cat.quantile(bins)
  print(quants)
  bins = list(quants[col])
  bins[0] = 0
  # labels = [1, 2, 3, 4, 5]
  labels = [1, 2, 3]
  df_cat = df_cat.apply(lambda x: pd.cut(x, bins=bins, labels=labels), axis=0)
  df_cat = df_cat.reset_index()
  return df_cat

dfgw_cat = dfw_cat(dfw , 'gwu_rate',1)
dfsw_cat = dfw_cat(dfw , 'swu_rate',0)

In [ ]:
fsvi = '<DATA_ROOT>/SVI/SVI_timeseries.csv'
df_svi = pd.read_csv(fsvi)
df_svi = df_svi.sort_values(by=['FIPS'])
formatted_numbers = [str(n).zfill(5) for n in df_svi['FIPS'].values]
df_svi['FIPS'] = formatted_numbers

# df_svi_cat = df_svi.drop(columns=['State','County']).set_index(['FIPS','year'])
# bins = [0, 0.25, 0.5, 0.75, 1]
# labels = [1, 2, 3, 4]
# df_svi_cat = df_svi_cat.apply(lambda x: pd.cut(x, bins=bins, labels=labels), axis=0)
# df_svi_cat = df_svi_cat['RPL_THEMES'].reset_index()

df_svi_cat = df_svi.drop(columns=['State','County']).set_index(['FIPS','year'])
bins = list(np.arange(0,101,5)/100)
labels = list(np.arange(0,100,5)/100)
df_svi_cat = df_svi_cat.apply(lambda x: pd.cut(x, bins=bins, labels=labels), axis=0)
df_svi_cat = df_svi_cat['RPL_THEMES'].reset_index()

In [ ]:
ffail = '<DATA_ROOT>/crop_failure_data/crops_failure_0595.csv'
df_fail = pd.read_csv(ffail)
df_fail = df_fail.drop(columns=['Unnamed: 0'])
df_fail = df_fail.sort_values(by=['FIPS'])
cond1 = df_fail['Planted Acres'] > 0
df_fail = df_fail[cond1]
df_fail['fail_share'] = df_fail['Failed Acres'] /  df_fail['Planted Acres']
df_fail = df_fail[df_fail.fail_share < 1]
# df_fail = df_fail[df_fail.fail_share > 0]

formatted_numbers = [str(n).zfill(5) for n in df_fail['FIPS'].values]
df_fail['FIPS'] = formatted_numbers


**Groundwater BoxPlot**

In [ ]:
cond_irig = df_fail['Irrigation Practice'] == 'I'
cols = ['FIPS','Crop','Planted Acres','year','fail_share']

df_merge = pd.merge(df_fail.loc[cond_irig,cols],dfgw_cat,on=['FIPS','year'])
df_merge = pd.merge(df_merge,df_svi_cat,on=['FIPS','year'])
df_merge['fail_event'] = (df_merge['fail_share'] > 0).astype(int)
df_merge

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 12))#,dpi=1200)
box_colors = ['#fff6ec','#fbd49b','#ed654d','#bf1e17']
# for i in [1,2,3,4,5]:
for i in [1,2,3]:
  df_filter = df_merge[df_merge['gwu_rate'] == i]
  print(i , len(df_filter))
  sns.boxplot(x='RPL_THEMES', y='fail_share', data=df_filter, ax=axes[i-1], whis=[1,99],palette=box_colors)
  # sns.swarmplot(x='RPL_THEMES', y='fail_share', data=df_filter, ax=axes[i-1], size=3 ,color="0.15",alpha=0.5)
  axes[i-1].set_ylim(0,0.5)
  axes[i-1].set_xticklabels(['Low','Low-Medium','Medium-High','High'])
  axes[i-1].set_xlabel('')
  axes[i-1].set_ylabel('Failure Share Intensity')

axes[i-1].set_xlabel('Social Vulnerability Index')

# out_path = '<DATA_ROOT>/Final_Exports/20231125-Section5/'
# plt.savefig(out_path + 'BoxPlot-SwarmPlot-GroundwaterUsage-FailureShare.png',dpi=1200)

**Surfacewater BoxPlot**

In [ ]:
cond_irig = df_fail['Irrigation Practice'] == 'I'
cols = ['FIPS','Crop','Planted Acres','year','fail_share']

df_merge = pd.merge(df_fail.loc[cond_irig,cols],dfsw_cat,on=['FIPS','year'])
df_merge = pd.merge(df_merge,df_svi_cat,on=['FIPS','year'])
df_merge['fail_event'] = (df_merge['fail_share'] > 0).astype(int)
df_merge

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 12),dpi=1200)
box_colors = ['#fff6ec','#fbd49b','#ed654d','#bf1e17']
# for i in [1,2,3,4,5]:
for i in [1,2,3]:
  df_filter = df_merge[df_merge['swu_rate'] == i]
  sns.boxplot(x='RPL_THEMES', y='fail_share', data=df_filter, ax=axes[i-1], whis=[1,99],palette=box_colors)
  sns.swarmplot(x='RPL_THEMES', y='fail_share', data=df_filter, ax=axes[i-1], size=3 ,color="0.15",alpha=0.5)
  axes[i-1].set_ylim(0,0.5)
  axes[i-1].set_xticklabels(['Low','Low-Medium','Medium-High','High'])
  axes[i-1].set_xlabel('')
  axes[i-1].set_ylabel('Failure Share Intensity')

axes[i-1].set_xlabel('Social Vulnerability Index')

# out_path = '<DATA_ROOT>/Final_Exports/20231125-Section5/'
# plt.savefig(out_path + 'BoxPlot-SwarmPlot-GroundwaterUsage-FailureShare.png',dpi=1200)

**Histogram GWUR(SWUR) vs SVI**

In [ ]:
fsvi = '<DATA_ROOT>/SVI/SVI_timeseries.csv'
df_svi = pd.read_csv(fsvi)
df_svi = df_svi.sort_values(by=['FIPS'])
formatted_numbers = [str(n).zfill(5) for n in df_svi['FIPS'].values]
df_svi['FIPS'] = formatted_numbers

df_svi_cat = df_svi.drop(columns=['State','County']).set_index(['FIPS','year'])
bins = list(np.arange(0,101,5)/100)
labels = list(np.arange(0,100,5)/100)
df_svi_cat = df_svi_cat.apply(lambda x: pd.cut(x, bins=bins, labels=labels), axis=0)
df_svi_cat = df_svi_cat['RPL_THEMES'].reset_index()

In [ ]:
cond_gu = dfw['swu_rate'] > 0
df_merge = pd.merge(dfw[cond_gu],df_svi_cat,on=['FIPS','year'])
df_group = df_merge.groupby('RPL_THEMES').agg({'swu_rate':'mean'})
df_group.iloc[1:].rolling(window=5,min_periods=1).mean().plot()
# plt.bar(df_group.index,df_group.gwu_rate)
# plt.ylim(bottom=0)
# plt.xlim(left=0,right=1)

# **Map Plot**

In [40]:
import xarray as xr
import pandas as pd
import numpy as np
import geopandas as gpd
from matplotlib import pyplot as plt
import seaborn as sns
from glob import glob

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)
warnings.simplefilter(action='ignore', category=RuntimeWarning)


In [ ]:
import pandas as pd
import numpy as np
fpath_water = '<DATA_ROOT>/WaterUsage/WaterUse_crop_v3.csv'
dfw = pd.read_csv(fpath_water).rename(columns={'Year':'year'})
dfw = dfw[['FIPS','IR-WGWFr','IR-WSWFr','IR-WFrTo','IR-IrTot','year']]
dfw['IR-WGWFr'] = pd.to_numeric(dfw['IR-WGWFr'], errors='coerce')
dfw['IR-WSWFr'] = pd.to_numeric(dfw['IR-WSWFr'], errors='coerce')
dfw['IR-IrTot'] = pd.to_numeric(dfw['IR-IrTot'], errors='coerce')
dfw = dfw[dfw['IR-IrTot'] > 0]
dfw.loc[:,'wu_ratio'] = (dfw['IR-WGWFr'] > dfw['IR-WSWFr']).astype(int)
dfw.loc[:,'gwu_rate'] = dfw['IR-WGWFr'] / dfw['IR-IrTot']
dfw.loc[:,'swu_rate'] = dfw['IR-WSWFr'] / dfw['IR-IrTot']
formatted_numbers = [str(n).zfill(5) for n in dfw['FIPS'].values]
dfw['FIPS'] = formatted_numbers
dfw = dfw[['FIPS','year','gwu_rate','swu_rate','wu_ratio']]

dfw_mean = dfw.groupby(['FIPS']).mean()[['gwu_rate','swu_rate']]
dfw_mean

In [ ]:
shp_counties = '<DATA_ROOT>/SVI/Counties/'
counties = gpd.read_file(shp_counties)
counties.iloc[0:5,:]

In [ ]:
ClimReg_dir = '<DATA_ROOT>/US_ClimateRegions/US_ClimateRegions/'
gdf_clim = gpd.read_file(ClimReg_dir)
gdf_clim

In [ ]:
states_dir = '<DATA_ROOT>/CONUS_States/'
gdf_states = gpd.read_file(states_dir)
gdf_states

In [78]:
from matplotlib.colors import ListedColormap, BoundaryNorm
colors = ['#ffffcc','#c7e9b4','#7fcdbb','#41b6c4','#2c7fb8','#253494']
cmap = ListedColormap(colors)
range_values = [0,0.04,0.2,0.4,0.8,1.4,24]
norm = BoundaryNorm(range_values, len(colors))

In [89]:
cnt_dfw_mean = pd.merge(counties,dfw_mean,on=['FIPS'])

fig , ax = plt.subplots(1,1,dpi=1200)
cnt_dfw_mean.plot(ax=ax,column='gwu_rate',cmap=cmap,norm=norm,legend=True,
                  legend_kwds={'cax': plt.axes([0.8, 0.25, 0.025, 0.2])})
counties.plot(ax=ax, color='none', edgecolor='black', alpha=0.2,linewidth=0.2)
gdf_states.plot(ax=ax, color='none', edgecolor='black', alpha=0.2,linewidth=0.6)
gdf_clim.plot(ax=ax, color='none', edgecolor='black', alpha=0.2,linewidth=1)

ax.set_xticks([])
ax.set_yticks([])

out_path = '<DATA_ROOT>/Final_Exports/20231125-Section5/'
plt.savefig(out_path + 'Map_Counties_GroundwaterUsageRate' + '.png' , dpi=1200)
plt.close()

In [90]:
cnt_dfw_mean = pd.merge(counties,dfw_mean,on=['FIPS'])


fig , ax = plt.subplots(1,1,dpi=1200)
cnt_dfw_mean.plot(ax=ax,column='swu_rate',cmap=cmap,norm=norm,legend=True,
                  legend_kwds={'cax': plt.axes([0.8, 0.25, 0.025, 0.2])})
counties.plot(ax=ax, color='none', edgecolor='black', alpha=0.2,linewidth=0.2)
gdf_states.plot(ax=ax, color='none', edgecolor='black', alpha=0.2,linewidth=0.6)
gdf_clim.plot(ax=ax, color='none', edgecolor='black', alpha=0.2,linewidth=1)

ax.set_xticks([])
ax.set_yticks([])

out_path = '<DATA_ROOT>/Final_Exports/20231125-Section5/'
plt.savefig(out_path + 'Map_Counties_SurfaceWaterUsageRate' + '.png' , dpi=1200)
plt.close()

# **TreeMap**

In [ ]:
df_merge

In [ ]:
groups = df_merge.groupby(['Crop','Irrigation Practice','water_usage','RPL_THEMES'])
df_group = groups.agg({'fail_share': ['mean', 'count']})
df_group.columns = [f'{col[0]}_{col[1]}' for col in df_group.columns]
df_group = df_group.reset_index()

svi_wu_vals = list(zip(df_group.water_usage.values,df_group.RPL_THEMES.values))
df_group['wu_svi'] = ['S'+str(s) if w==0 else 'G'+str(s) for w,s in svi_wu_vals]
df_group

In [ ]:
df_group

In [ ]:
import plotly.express as px

reds = ['#fff6ec','#fbd49b','#ed654d','#820f0a']
blues = ['#eff3ff','#bdd7e7','#6baed6','#2171b5']
wu_svi_colors = {
    'S1': blues[0], 'S2':blues[1], 'S3':blues[2], 'S4':blues[3],
    'G1': reds[0], 'G2': reds[1], 'G3': reds[2], 'G4': reds[3]
}

df = df_group[(df_group['Irrigation Practice'] == 'I')]
# df = df[(df['Crop'] == 'CORN')]

fig = px.treemap(df,
                 path=['Crop', 'water_usage', 'RPL_THEMES'], #
                 #px.Constant("US"),'ClimRegNam','Irrigation Practice', 'RPL_THEMES'
                 values='fail_share_count',
                 color='wu_svi',
                 color_discrete_map=wu_svi_colors
                 )
# colors_crop = {
#     'WHEAT': '#add8e6',       # Light Blue for young wheat, turning to '#f5deb3' for mature wheat
#     'CORN': '#008000',        # Dark Green for corn
#     'SOYBEANS': '#00ff00',    # Bright Green for soybeans
#     'COTTON': '#bdbd37',      # Light Yellow for cotton
#     'SORGHUM': '#ff4500',     # Reddish-Orange for sorghum
#     'HAY': '#785705',         # Dark Goldenrod for hay
#     'OATS': '#ccc9c4',        # Bisque for oats
#     'BARLEY': '#f56c1d'       # Sandy Brown for barley
# }

# clim_colors = ['#fcfbfd','#efedf5','#dadaeb','#bcbddc','#9e9ac8','#807dba','#6a51a3','#54278f','#3f007d']
# clim_reg_colors = dict(zip(list(df_group.ClimRegNam.unique()),clim_colors))
# fig.for_each_trace(
#     lambda t: t.update(
#         marker_colors=[
#             clim_reg_colors[id.split("/")[0]] if len(id.split("/")) == 1 else c
#             for c, id in zip(t.marker.colors, t.ids)
#         ]
#     )
# )

# fig.update_layout(
#     width=1500,  # Set the width of the figure in pixels
#     height=1200,  # Set the height of the figure in pixels
# )

fig.show()

# **Density Plot**

In [ ]:
import pandas as pd
import numpy as np
fpath_water = '<DATA_ROOT>/WeatherIndex/Water_use_2009_2023.csv'
dfw = pd.read_csv(fpath_water)
dfw = dfw.drop(columns=['STATE' ,	'COUNTY']).rename(columns={'Year':'year'})
dfw['IC-WGWFr'] = pd.to_numeric(dfw['IC-WGWFr'], errors='coerce')
dfw['IC-WSWFr'] = pd.to_numeric(dfw['IC-WSWFr'], errors='coerce')
dfw['IC-WFrTo'] = pd.to_numeric(dfw['IC-WFrTo'], errors='coerce')
dfw = dfw.dropna()
formatted_numbers = [str(n).zfill(5) for n in dfw['FIPS'].values]
dfw['FIPS'] = formatted_numbers
dfw = dfw[dfw['IC-WFrTo'] > 0]
dfw['water_usage'] = (dfw['IC-WGWFr'] > dfw['IC-WSWFr']).astype(int)
# dfw['gr_water_usage'] = dfw['IC-WGWFr'] / dfw['IC-WFrTo']
# dfw['su_water_usage'] = dfw['IC-WSWFr'] / dfw['IC-WFrTo']
# dfw['ratio_water_usage'] = dfw['IC-WSWFr'] / dfw['IC-WGWFr']
dfw = dfw[['FIPS','year','water_usage']]#,'ratio_water_usage','gr_water_usage','su_water_usage'
# dfw.replace([np.inf, -np.inf], np.nan, inplace=True)
# dfw = dfw.dropna()
dfw

In [ ]:
ffail = '<DATA_ROOT>/crop_failure_data/crops_failure_0595.csv'
df_fail = pd.read_csv(ffail)
df_fail = df_fail.drop(columns=['Unnamed: 0'])
df_fail = df_fail.sort_values(by=['FIPS'])
# cond1 = df_fail['year']>2011
cond2 = df_fail['Planted Acres'] > 0
df_fail = df_fail[cond2]
df_fail['fail_share'] = df_fail['Failed Acres'] /  df_fail['Planted Acres']
df_fail = df_fail[df_fail.fail_share < 1]
# df_fail = df_fail[df_fail.fail_share > 0]

formatted_numbers = [str(n).zfill(5) for n in df_fail['FIPS'].values]
df_fail['FIPS'] = formatted_numbers

# df_fail.year.unique()

In [ ]:
fpath_counties_clim = '<DATA_ROOT>/US_ClimateRegions/Counties_ClimReg.csv'
df_counties_clim = pd.read_csv(fpath_counties_clim)
df_counties_clim = df_counties_clim.drop(columns=['Unnamed: 0',	'State', 	'County'])
formatted_numbers = [str(n).zfill(5) for n in df_counties_clim['FIPS'].values]
df_counties_clim['FIPS'] = formatted_numbers
df_counties_clim

In [ ]:
dfw_clim = pd.merge(dfw,df_counties_clim,on=['FIPS'])
# dfw_clim

In [ ]:
df_svi_cat = df_svi.drop(columns=['State', 'County'])
df_svi_cat = df_svi_cat.set_index(['FIPS','year'])
bins = list(np.arange(0,101,5)/100)
labels = list(np.arange(0,100,5)/100)
# bins = [0, 0.25, 0.5, 0.75, 1]
# labels = [1, 2, 3, 4]
df_svi_cat = df_svi_cat.apply(lambda x: pd.cut(x, bins=bins, labels=labels), axis=0)
df_svi_cat = df_svi_cat.reset_index()[['FIPS','year','RPL_THEMES']]
df_svi_cat

In [ ]:
df_merge = pd.merge(df_fail,dfw,on=['FIPS','year'])
df_merge = pd.merge(df_merge,df_svi_cat,on=['FIPS','year'])
df_merge = df_merge.dropna()
df_merge

cols_merge = ['Crop','Planted Acres','year','fail_share','RPL_THEMES','water_usage']#,'FIPS',
cond_ig = df_merge['Irrigation Practice'] == 'N'
df_merge = df_merge[cond_ig][cols_merge]
df_merge

In [ ]:
cond1 = df_merge['water_usage'] == 0
theme = 'RPL_THEMES'
df_CropAreaTheme = df_merge[cond1].groupby(['Crop',theme , 'year'])['Planted Acres'].sum().reset_index()
df_CropAreaTheme = df_CropAreaTheme.groupby(['Crop',theme])['Planted Acres'].mean().reset_index()
df_CropAreaTheme = df_CropAreaTheme.pivot(index=theme,columns='Crop',values='Planted Acres')

df_CropFailTheme = df_merge[cond1].groupby(['Crop',theme]).mean().reset_index()
df_CropFailTheme = df_CropFailTheme.pivot(index=theme,columns='Crop',values='fail_share')


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter
import seaborn as sns
import numpy as np

crops = ['CORN', 'COTTON', 'OATS', 'SORGHUM', 'SOYBEANS', 'WHEAT', 'BARLEY', 'HAY']
fig, axes = plt.subplots(nrows=4, ncols=2,figsize=(16,12),dpi=1200)
i = 0

for ax_row in axes:
    for ax in ax_row:

        ws = 4
        sns.lineplot(data=df_CropAreaTheme.rolling(window=ws, min_periods=1).mean(),
                     x=df_CropAreaTheme.index, y=crops[i], color='#3ec429', ax=ax)
        ax.fill_between(df_CropAreaTheme.index, 0,
                        df_CropAreaTheme[crops[i]].rolling(window=ws, min_periods=1).mean(),
                        color='#3ec429', alpha=0.3)

        ax_twin = ax.twinx()
        sns.lineplot(data=df_CropFailTheme.rolling(window=ws, min_periods=1).mean(),
                     x=df_CropFailTheme.index, y=crops[i], color='#e69447', ax=ax_twin)
        ax_twin.fill_between(df_CropFailTheme.index, 0,
                             df_CropFailTheme[crops[i]].rolling(window=ws, min_periods=1).mean(),
                             color='#e69447', alpha=0.3)

        ax.set_title(crops[i])
        ax.set_xticks([])
        # ax.set_yticks([])
        ax.set_xlabel('')
        ax.set_ylabel('')
        ax.set_ylim(0)
        ax.set_xlim(0,1)


        if i > 5:
          ax.set_xticks(list(np.arange(0,11,2)/10))
          ax.set_xticklabels(list(np.arange(0,11,2)/10), fontsize=16)


        # ax_twin.set_yticks([])
        ax_twin.set_xlabel('')
        ax_twin.set_ylabel('')
        ax_twin.set_ylim(0)
        ax_twin.set_xlim(0,1)

        i += 1

plt.suptitle('Rainfed ' + 'Surfacewater',fontsize=14)
out_path = '<DATA_ROOT>/Final_Exports/20231125-Section5/20231125-DensityPlot-Ground-Surface-Water/'
plt.savefig(out_path + 'DensityPlot_Rainfed_Surfacewater' + '.png' , dpi=1200)
# plt.tight_layout()
# plt.show()


In [ ]:
df_fail = df_fail[['FIPS','Crop','Irrigation Practice','Planted Acres','year','fail_share']]
df_fail['fail_share'] = df_fail['fail_share'] / df_fail['fail_share']
df_fail = df_fail.fillna(0)
df_fail

In [ ]:
# group = df_merge.groupby(['ClimRegNam','Irrigation Practice','Crop','RPL_THEMES','water_usage'])
# df_group = group[['Planted Acres','fail_share']].sum()
# df_group = df_group.reset_index()
# df_group['fail_rate'] = (df_group['fail_share'] / df_group['Planted Acres']) * 1e5
# df_group
# df_group.to_csv('/content/df_group3.csv')

In [ ]:
df_merge.Crop.unique()

In [ ]:
cond_ig = df_merge['Irrigation Practice'] == 'I'
group = df_merge[cond_ig].groupby(['Crop','RPL_THEMES'])#,'gr_water_usage','ClimRegNam'
df_group = group[['Planted Acres','fail_share']].sum()
df_group['fail_rate'] = (df_group['fail_share'] / df_group['Planted Acres']) * 1e5
df_group['gr_water_usage'] = group[['gr_water_usage']].mean()
# df_group.to_csv('/content/df_group3.csv')
df_group = df_group.reset_index()
df_group

df_F_rate = df_group.pivot(index='RPL_THEMES',columns='Crop',values='fail_share')
df_G_mean = df_group.pivot(index='RPL_THEMES',columns='Crop',values='gr_water_usage')
df_G_mean

In [ ]:
df

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter
import seaborn as sns
import numpy as np

clims = ['Northeast', 'Northern Rockies and Plains', 'Northwest', 'Ohio Valley',
       'South', 'Southeast', 'Southwest', 'Upper Midwest', 'West']
crops = ['CORN', 'COTTON', 'OATS', 'SORGHUM', 'SOYBEANS', 'WHEAT', 'BARLEY',
       'HAY']
fig, axes = plt.subplots(nrows=4, ncols=2,figsize=(16,16),dpi=600)
i = 0

for ax_row in axes:
    for ax in ax_row:
      if i == 8:
        break

      ws = 4
      sns.lineplot(data=df_G_mean.rolling(window=ws, min_periods=1).mean(),
                    x=df_G_mean.index, y=crops[i], color='#3ec429', ax=ax)
      ax.fill_between(df_G_mean.index, 0,
                      df_G_mean[crops[i]].rolling(window=ws, min_periods=1).mean(),
                      color='#3ec429', alpha=0.3)

      ax_twin = ax.twinx()
      sns.lineplot(data=df_F_rate.rolling(window=ws, min_periods=1).mean(),
                    x=df_F_rate.index, y=crops[i], color='#e69447', ax=ax_twin)
      ax_twin.fill_between(df_F_rate.index, 0,
                            df_F_rate[crops[i]].rolling(window=ws, min_periods=1).mean(),
                            color='#e69447', alpha=0.3)

      ax.set_title(crops[i])
      ax.set_xticks([])
      # ax.set_yticks([])
      ax.set_xlabel('')
      ax.set_ylabel('')
      ax.set_ylim(0)
      ax.set_xlim(0,1)


      # if i > 6:
      #   ax.set_xticks(list(np.arange(0,101)/100))
      #   ax.set_xticklabels(list(np.arange(0,101)/100), fontsize=16)


      # ax_twin.set_yticks([])
      ax_twin.set_xlabel('')
      ax_twin.set_ylabel('')
      ax_twin.set_ylim(0)
      ax_twin.set_xlim(0,1)

      i += 1

# plt.suptitle(f'{irig_type} {theme}',fontsize=14)
# out_path = '<DATA_ROOT>/Final_Exports/20231120-Section2/20231121-DensityPlot_FailurePlant_SVI_ALL-Irrigation/'
# plt.savefig(out_path + 'DensityPlot_FailurePlant_' + theme + '_' + "".join(irig_type.split()) + '.png' , dpi=1200)
# plt.tight_layout()
# plt.show()


In [ ]:
cond_ig = df_merge['Irrigation Practice'] == 'I'
group = df_merge[cond_ig].groupby(['RPL_THEMES'])#,'gr_water_usage','ClimRegNam',
df_group = group[['Planted Acres','fail_share']].sum()
df_group['fail_rate'] = (df_group['fail_share'] / df_group['Planted Acres']) * 1e5
df_group['gr_water_usage'] = group[['gr_water_usage']].mean()
# df_group.to_csv('/content/df_group3.csv')
df_group #= df_group.reset_index()
df_group = df_group[['fail_rate' ,	'gr_water_usage']]#'fail_share' ,

fig, ax = plt.subplots(figsize=(16,8),dpi=300)
ws = 20
sns.lineplot(data=df_group['gr_water_usage'].rolling(window=ws, min_periods=5).mean(),
              x=df_group.index, y=df_group['gr_water_usage'], color='#3ec429', ax=ax)
ax.fill_between(df_group.index, 0,
                df_group['gr_water_usage'].rolling(window=ws, min_periods=5).mean(),
                color='#3ec429', alpha=0.3)

ax_twin = ax.twinx()
sns.lineplot(data=df_group['fail_rate'].rolling(window=ws, min_periods=5).mean(),
              x=df_group.index, y=df_group['fail_rate'], color='#e69447', ax=ax_twin)
ax_twin.fill_between(df_group.index, 0,
                      df_group['fail_rate'].rolling(window=ws, min_periods=5).mean(),
                      color='#e69447', alpha=0.3)

In [ ]:
df_group.columns

In [ ]:
# cond_I = df_merge['Irrigation Practice'] == 'I'
# cond_N = df_merge['Irrigation Practice'] == 'N'
# cond_su = df_merge['water_usage'] == 0
# cond_gr = df_merge['water_usage'] == 1

# df_su_I = df_merge[cond_I & cond_su]
# df_su_N = df_merge[cond_N & cond_su]
# df_gr_I = df_merge[cond_I & cond_gr]
# df_gr_N = df_merge[cond_N & cond_gr]

# df_grp_suI = df_su_I.groupby(['RPL_THEMES']).sum()['Planted Acres'] / 1e6
# df_grp_suN = df_su_N.groupby(['RPL_THEMES']).sum()['Planted Acres'] / 1e6
# df_grp_grI = df_gr_I.groupby(['RPL_THEMES']).sum()['Planted Acres'] / 1e6
# df_grp_grN = df_gr_N.groupby(['RPL_THEMES']).sum()['Planted Acres'] / 1e6

# df_wu_I = df_merge[cond_clim & cond_I].groupby(['RPL_THEMES']).mean()[['Planted Acres','gr_water_usage','su_water_usage']]
# df_wu_N = df_merge[cond_clim & cond_N].groupby(['RPL_THEMES']).mean()[['Planted Acres','gr_water_usage','su_water_usage']]

# df_wu_I['Planted Acres'] = df_wu_I['Planted Acres'] / 1e6
# df_wu_N['Planted Acres'] = df_wu_N['Planted Acres'] / 1e6
# df_wu_I

list_df = []
for clim in list_clim:
  cond_clim = df_merge['ClimRegNam'] == clim
  cond_I = df_merge['Irrigation Practice'] == 'I'
  df_wu_I = df_merge[cond_clim & cond_I].groupby(['RPL_THEMES']).mean()[['Planted Acres','gr_water_usage','su_water_usage']]
  list_df.append(df_wu_I)
  print(clim , len(df_merge[cond_clim & cond_I]))


In [ ]:
# df_grp_suI.plot(color='blue')
# df_grp_suN.plot(color='red')
# df_grp_grI.plot(color='green')
# df_grp_grN.plot(color='yellow')

# df_wu_I.rolling(window=20,min_periods=5).mean().gr_water_usage.plot(color='red')
# df_wu_I.rolling(window=20,min_periods=5).mean().su_water_usage.plot(color='blue')
# df_wu_I.su_water_usage.plot(color='red')
# df_wu_N.gr_water_usage.plot(color='green')
# df_wu_N.su_water_usage.plot(color='yellow')
['Southeast', 'Southwest', 'South', 'West', 'Northeast',
       'Northwest', 'Ohio Valley', 'Upper Midwest',
       'Northern Rockies and Plains']
colors = ['#e41a1c','#377eb8','#4daf4a','#984ea3','#ff7f00','#ffff33','#a65628','#f781bf','#999999']
for i , df in enumerate(list_df):
  df = list_df[4]
  df.gr_water_usage.rolling(window=20,min_periods=5).mean().plot(color=colors[i])#
  break